# 03 - Análisis por rama: empleo, salarios, producción y exposición a importaciones

## Preguntas
1. ¿Cuántos puestos de trabajo perdió (o ganó) cada rama industrial, y cómo cambiaron sus salarios reales y su producción?
2. ¿Las ramas cuyas importaciones crecieron más (o donde más avanzó China) tuvieron una evolución distinta de las demás?
3. Controlando por lo que le pasa a toda la industria cada año, ¿se mueven juntos importaciones y empleo/producción/salarios *dentro* de cada rama?

## Método (y por qué)
- **Ventanas de comparación:** promedio del año base (2018 o 2023) contra el promedio de los **últimos 12 meses
  disponibles** (4 trimestres en empleo, último año en empresas). Comparar meses sueltos engaña: el IPI tiene
  estacionalidad fuerte y las remuneraciones incluyen aguinaldo en junio y diciembre.
- **Exposición a importaciones:** crecimiento de las importaciones entre el año base y 2025 (en dólares y en
  toneladas), participación de China y su aumento en puntos porcentuales.
- **Correlación de Spearman** (por rangos): con 16 ramas y valores muy dispares, la de Pearson la domina un solo punto.
- **Panel con efectos fijos de rama y de año**: `log(resultado) ~ log(importaciones)`. Los efectos fijos de año
  absorben lo que le pasa a toda la industria a la vez (devaluaciones, recesiones); los de rama, las diferencias
  permanentes. Errores estándar agrupados por rama.

## Limitaciones que hay que tener presentes
- Son **asociaciones, no efectos causales**. Las importaciones de una rama suben también cuando esa rama crece
  (insumos, bienes de capital): eso puede mostrar una relación *positiva* entre importaciones y empleo que no
  significa que importar cree empleo. Separar importaciones que compiten de las que complementan requeriría la
  clasificación por uso económico (BEC), que no está en estos archivos.
- n = 16 ramas: pocas observaciones. Se prueban muchas combinaciones: con 24 correlaciones, ~1 sale "significativa" por azar.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import analysis as A
import loaders as L
from ramas import RAMA_LABELS, TOTAL

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)

In [2]:
def leer(nombre, **kw):
    return pd.read_csv(PROCESSED / nombre, dtype={"rama": str}, **kw)

E = leer("empleo_por_rama.csv", parse_dates=["periodo"])
S = leer("salario_real_por_rama.csv", parse_dates=["periodo"])
I = leer("ipi_por_rama.csv", parse_dates=["periodo"])
C = leer("empresas_por_rama.csv", parse_dates=["periodo"])
imp_rp = leer("importaciones_rama_pais_anio.csv")
exp_r = leer("exportaciones_rama_anio.csv")
ipi_full = L.load_ipi_manufacturero()

## 1. Series suavizadas e indexadas (promedio 2023 = 100)

In [3]:
series = A.series_indexadas(E, S, I)
series.to_csv(PROCESSED / "series_indexadas.csv", index=False)
series[series.rama == TOTAL].groupby("metrica").tail(1)[["metrica", "periodo", "indice"]].round(1)

,metrica,periodo,indice
2005,Empleo,2026-01-01,95.8
3722,Salario real,2026-03-01,102.4
5694,Producción (IPI),2026-07-01,90.6


## 2. Cambios por rama: puestos, salario real, producción, empresas

In [4]:
cambios = A.cambios_por_rama(E, S, I, C)
cambios.to_csv(PROCESSED / "cambios_por_rama.csv", index=False)
cambios[cambios.rama == TOTAL][["ventana", "metrica", "base", "ultimo", "delta_abs", "delta_pct", "fin"]].round(1)

,ventana,metrica,base,ultimo,delta_abs,delta_pct,fin
16,2018,Empleo,1219283.5,1216127.8,-3155.8,-0.3,1T 2026
33,2023,Empleo,1268930.0,1216127.8,-52802.2,-4.2,1T 2026
50,2018,Salario real,25975.8,23914.1,-2061.7,-7.9,mar 2026
67,2023,Salario real,23360.1,23914.1,554.1,2.4,mar 2026
84,2018,Producción (IPI),125.9,117.0,-8.9,-7.1,jul 2026
101,2023,Producción (IPI),129.2,117.0,-12.2,-9.4,jul 2026
118,2018,Empresas,55591.0,54070.0,-1521.0,-2.7,2024
135,2023,Empresas,54970.0,54070.0,-900.0,-1.6,2024


In [5]:
# empleo: último trimestre contra el mismo trimestre de los años base (sin estacionalidad)
q = A.empleo_mismo_trimestre(E)
q.to_csv(PROCESSED / "empleo_mismo_trimestre.csv", index=False)
q[q.rama == TOTAL].round(1)

,rama,base,ultimo,delta_abs,delta_pct,ventana,trimestre_base,trimestre_ultimo
16,TOTAL,1246245,1200862,-45383,-3.6,2018,1T 2018,1T 2026
33,TOTAL,1265769,1200862,-64907,-5.1,2023,1T 2023,1T 2026


### Ramas que más puestos perdieron y ganaron (base 2018)

La industria en su conjunto casi no cambió desde 2018, pero eso esconde una reasignación: unas ramas perdieron mucho
y otras ganaron.

In [6]:
emp18 = cambios[(cambios.metrica == "Empleo") & (cambios.ventana == "2018") & (cambios.rama != TOTAL)]
(emp18.assign(rama_label=emp18.rama.map(RAMA_LABELS)).sort_values("delta_abs")
 [["rama_label", "base", "ultimo", "delta_abs", "delta_pct"]].round(1))

,rama_label,base,ultimo,delta_abs,delta_pct
3,"Prendas de vestir, cuero y calzado",74105.2,58271.8,-15833.5,-21.4
4,"Madera, papel, edición e impresión",98223.5,86083.5,-12140.0,-12.4
2,Textiles (hilados y tejidos),61604.2,55499.8,-6104.5,-9.9
8,Minerales no metálicos,46664.2,43838.5,-2825.8,-6.1
9,Metálicas básicas,36557.2,35339.8,-1217.5,-3.3
7,Caucho y plástico,65055.5,63882.2,-1173.2,-1.8
5,Refinación de petróleo,5678.0,4742.5,-935.5,-16.5
13,Automotores y autopartes,75740.8,75152.0,-588.8,-0.8
1,Tabaco,5255.5,4752.5,-503.0,-9.6
14,Otro equipo de transporte,10320.2,10226.2,-94.0,-0.9


## 3. Detalle de producción por subrama (calzado, muebles, informática...)

In [7]:
nombres, sub_idx, sub_cambios = A.ipi_subramas(ipi_full)
nombres.to_csv(PROCESSED / "ipi_subramas_nombres.csv", index=False)
sub_idx.to_csv(PROCESSED / "ipi_subramas_indexadas.csv", index=False)
sub_cambios.to_csv(PROCESSED / "ipi_subramas_cambios.csv", index=False)
(sub_cambios[sub_cambios.ventana == "2023"].merge(nombres, on="codigo")
 .sort_values("delta_pct")[["nombre", "delta_pct", "fin"]].head(8).round(1))

,nombre,delta_pct,fin
35,Neumáticos,-48.9,ene 2026
17,Tejidos y acabado de productos textiles,-46.5,jul 2026
54,"Equipos y aparatos de informática, televisión ...",-42.7,jul 2026
53,Aparatos de uso doméstico,-37.2,jul 2026
39,Productos de arcilla y cerámica no refractaria,-37.0,jul 2026
21,Calzado y sus partes,-36.6,jul 2026
15,Preparación de fibras de uso textil,-32.8,jul 2026
16,Hilados de algodón,-32.7,jul 2026


## 4. Comercio exterior por rama (2016-2025)

Importaciones (CIF y toneladas), importaciones desde China, exportaciones y balanza comercial por rama y año.

In [8]:
comercio = A.comercio_anual(imp_rp, exp_r)
comercio.to_csv(PROCESSED / "comercio_por_rama_anio.csv", index=False)
(comercio[comercio.anio.isin([2018, 2023, 2025])]
 .assign(rama_label=lambda d: d.rama.map(RAMA_LABELS), imp_cif_mm=lambda d: (d.imp_cif / 1e6).round(0), imp_mt=lambda d: (d.imp_kg / 1e9).round(2), pct_china=lambda d: d.pct_china.round(1))
 .pivot(index="rama_label", columns="anio", values=["imp_cif_mm", "pct_china"]))

imp_cif_mm           \
anio                                                     2018     2023   
rama_label                                                               
Alimentos y bebidas                                    1118.0   1156.0   
Automotores y autopartes                              10273.0   7504.0   
Caucho y plástico                                      3395.0   4400.0   
Informática, electrónica, eléctricos e instrume...    10764.0  11437.0   
Madera, papel, edición e impresión                     1320.0   1489.0   
Maquinaria y equipo                                    8538.0   9916.0   
Metálicas básicas                                      1979.0   2725.0   
Minerales no metálicos                                  797.0    803.0   
Muebles y otras manufacturas                           1075.0    959.0   
Otro equipo de transporte                              1249.0    934.0   
Prendas de vestir, cuero y calzado                     1323.0   1300.0   
Productos de metal                                     1643.0   1975.0   
Químicos (incl. farmacéuticos)                         9703.0  11833.0   
Refinación de petróleo                                 2954.0   3819.0   
Tabaco                                                   51.0     71.0   
Textiles (hilados y tejidos)                           1056.0   1138.0   

                                                            pct_china        \
anio                                                   2025      2018  2023   
rama_label                                                                    
Alimentos y bebidas                                  2016.0       2.7   3.4   
Automotores y autopartes                            13225.0       2.8   4.5   
Caucho y plástico                                    4337.0      12.1  18.0   
Informática, electrónica, eléctricos e instrume...  11629.0      47.3  46.4   
Madera, papel, edición e impresión                   1276.0       8.6  13.3   
Maquinaria y equipo                                 11191.0      18.8  25.6   
Metálicas básicas                                    2190.0       9.9  11.0   
Minerales no metálicos                                806.0      18.7  17.9   
Muebles y otras manufacturas                         1316.0      54.0  48.6   
Otro equipo de transporte                            1607.0      47.0  50.0   
Prendas de vestir, cuero y calzado                   1794.0      44.4  39.9   
Productos de metal                                   2006.0      23.5  24.0   
Químicos (incl. farmacéuticos)                      12364.0      15.7  19.7   
Refinación de petróleo                               1679.0       1.5   0.4   
Tabaco                                                 55.0       0.0   0.0   
Textiles (hilados y tejidos)                         1108.0      39.1  49.1   

                                                          
anio                                                2025  
rama_label                                                
Alimentos y bebidas                                  3.4  
Automotores y autopartes                             9.8  
Caucho y plástico                                   27.4  
Informática, electrónica, eléctricos e instrume...  46.4  
Madera, papel, edición e impresión                  19.3  
Maquinaria y equipo                                 28.4  
Metálicas básicas                                   15.1  
Minerales no metálicos                              24.7  
Muebles y otras manufacturas                        59.0  
Otro equipo de transporte                           54.1  
Prendas de vestir, cuero y calzado                  39.1  
Productos de metal                                  34.2  
Químicos (incl. farmacéuticos)                      19.0  
Refinación de petróleo                               0.6  
Tabaco                                               0.0  
Textiles (hilados y tejidos)                        53.2

## 5. Exposición vs. resultados

Una fila por rama y ventana: cuánto crecieron las importaciones (USD y toneladas), cuánto avanzó China, y cómo
evolucionaron empleo, salario real y producción.

In [9]:
base = A.exposicion_vs_resultados(cambios, comercio)
base.to_csv(PROCESSED / "exposicion_vs_resultados.csv", index=False)
base[base.ventana == "2023"].assign(rama_label=lambda d: d.rama.map(RAMA_LABELS)).set_index("rama_label")[
    ["d_imp_cif", "d_imp_kg", "d_china_pp", "d_empleo", "d_salario", "d_ipi"]].round(1).sort_values("d_imp_kg", ascending=False)

,d_imp_cif,d_imp_kg,d_china_pp,d_empleo,d_salario,d_ipi
rama_label,,,,,,
Alimentos y bebidas,74.4,149.9,-0.1,1.4,2.2,1.4
Muebles y otras manufacturas,37.3,102.8,10.4,-2.2,1.1,-9.6
"Prendas de vestir, cuero y calzado",38.0,71.1,-0.8,-14.9,2.9,-18.4
Automotores y autopartes,76.2,65.1,5.4,-5.6,5.1,-16.7
Otro equipo de transporte,72.0,64.2,4.1,-4.1,-1.3,10.8
Textiles (hilados y tejidos),-2.6,34.2,4.2,-12.1,-12.2,-34.5
Maquinaria y equipo,12.9,33.5,2.8,-4.1,4.7,-24.7
"Informática, electrónica, eléctricos e instrumentos",1.7,33.3,0.1,-7.6,2.2,-19.2
Minerales no metálicos,0.3,31.3,6.8,-10.3,-5.8,-23.7


In [10]:
corr = A.correlaciones(base)
corr.to_csv(PROCESSED / "correlaciones.csv", index=False)
corr.drop(columns=["exposicion_col", "resultado_col"]).round(3)

,ventana,exposicion,resultado,rho,p_valor,n_ramas,p_corregido
0,2018,Crecimiento de las importaciones (USD),Δ% empleo,0.506,0.046,16,0.365
1,2018,Crecimiento de las importaciones (USD),Δ% salario real,0.291,0.274,16,0.644
2,2018,Crecimiento de las importaciones (USD),Δ% producción (IPI),0.126,0.641,16,0.894
3,2018,Crecimiento de las importaciones (toneladas),Δ% empleo,0.376,0.151,16,0.480
4,2018,Crecimiento de las importaciones (toneladas),Δ% salario real,-0.088,0.745,16,0.894
5,2018,Crecimiento de las importaciones (toneladas),Δ% producción (IPI),-0.276,0.300,16,0.644
6,2018,Aumento de la participación de China (p.p.),Δ% empleo,0.065,0.812,16,0.900
7,2018,Aumento de la participación de China (p.p.),Δ% salario real,0.047,0.863,16,0.900
8,2018,Aumento de la participación de China (p.p.),Δ% producción (IPI),-0.265,0.322,16,0.644
9,2018,Participación de China en las importaciones (n...,Δ% empleo,0.091,0.737,16,0.894


`rho` cercano a -1: a más exposición, peor evolución; cercano a +1: a más exposición, mejor evolución.
`p_valor` bajo (menos de 0,05) indica que es poco probable que sea casualidad. Con 24 combinaciones probadas,
1 o 2 valores bajos pueden aparecer por azar: se prioriza lo que es consistente entre ventanas y medidas.

In [11]:
grupos = A.grupos_por_exposicion(base)
grupos.to_csv(PROCESSED / "grupos_exposicion.csv", index=False)
grupos.drop(columns="ramas").round(1)

,ventana,exposicion,grupo,n_ramas,mediana_d_empleo,mediana_d_salario,mediana_d_ipi,delta_puestos
0,2018,Crecimiento de las importaciones (USD),Mayor exposición,8,-0.4,-10.4,-11.5,20111.5
1,2018,Crecimiento de las importaciones (USD),Menor exposición,8,-7.8,-11.6,-19.9,-23267.2
2,2018,Crecimiento de las importaciones (toneladas),Mayor exposición,8,-0.4,-10.5,-20.1,11931.5
3,2018,Crecimiento de las importaciones (toneladas),Menor exposición,8,-4.7,-9.5,-9.1,-15087.2
4,2018,Aumento de la participación de China (p.p.),Mayor exposición,8,-1.4,-10.8,-15.6,-17727.0
5,2018,Aumento de la participación de China (p.p.),Menor exposición,8,-1.6,-10.5,-11.5,14571.2
6,2018,Participación de China en las importaciones (n...,Mayor exposición,8,-0.4,-11.0,-22.1,-17815.5
7,2018,Participación de China en las importaciones (n...,Menor exposición,8,-4.7,-9.0,-4.8,14659.8
8,2023,Crecimiento de las importaciones (USD),Mayor exposición,8,-4.1,2.6,-13.1,-18696.0
9,2023,Crecimiento de las importaciones (USD),Menor exposición,8,-7.5,0.3,-19.7,-34106.2


## 6. Panel con efectos fijos de rama y año

Aísla la relación *dentro de cada rama* entre importaciones y resultados, descontando lo que le pasa a toda la
industria cada año. `efecto_10pct` = variación % del resultado ante un aumento de 10% de las importaciones.

In [12]:
panel = A.panel_anual(E, S, I, comercio)
panel = panel[panel.anio >= 2016]
panel.to_csv(PROCESSED / "panel_rama_anio.csv", index=False)

regs = A.regresiones_panel(panel)
regs.to_csv(PROCESSED / "regresiones_panel.csv", index=False)
regs[["resultado", "importaciones", "muestra", "rezago_anios", "elasticidad", "p_valor", "efecto_10pct", "puestos_por_10pct", "n_obs"]].round(3)

,resultado,importaciones,muestra,rezago_anios,elasticidad,p_valor,efecto_10pct,puestos_por_10pct,n_obs
0,Empleo,Importaciones totales,Todas las ramas,0,0.023,0.639,0.219,2683.840,160
1,Empleo,Importaciones totales,Todas las ramas,1,0.010,0.798,0.093,1138.851,144
2,Empleo,Importaciones totales,Sin refinación de petróleo,0,0.082,0.000,0.778,9552.056,150
3,Empleo,Importaciones totales,Sin refinación de petróleo,1,0.054,0.009,0.519,6370.129,135
4,Empleo,Importaciones desde China,Todas las ramas,0,0.028,0.577,0.266,3262.154,160
5,Empleo,Importaciones desde China,Todas las ramas,1,0.014,0.787,0.133,1636.751,144
6,Empleo,Importaciones desde China,Sin refinación de petróleo,0,0.082,0.015,0.785,9638.237,150
7,Empleo,Importaciones desde China,Sin refinación de petróleo,1,0.076,0.003,0.728,8939.833,135
8,Producción (IPI),Importaciones totales,Todas las ramas,0,0.202,0.100,1.929,NaN,160
9,Producción (IPI),Importaciones totales,Todas las ramas,1,0.155,0.064,1.478,NaN,144


## Respuesta a la pregunta central (2018 → 2025)

**Desde 2018, ¿la industria argentina perdió empleo, producción y salario justo en los sectores donde más crecieron las importaciones?**

Ventana común 2018 → 2025 (último año completo en todas las fuentes). Medida principal: crecimiento % de las importaciones en dólares.
Las otras medidas (toneladas, importaciones por puesto) y las pruebas de sensibilidad sirven para ver si la respuesta cambia al medir distinto.

## Importaciones por uso económico (consumo, insumos y capital)

Cada producto importado (NCM, primeros 6 dígitos = HS) se clasifica con las tablas HS → BEC del Banco Mundial (WITS) según la versión del HS de cada año.
Separa lo que **compite** con la producción local (bienes de consumo) de lo que la **abastece** (insumos, partes y máquinas). Ver `src/uso_economico.py`.

In [13]:
import uso_economico as U

por_uso = pd.concat([U.importaciones_por_uso(L.load_importaciones(a), a) for a in range(2016, 2026)], ignore_index=True)
por_uso.to_csv(PROCESSED / "importaciones_por_uso.csv", index=False)
# control: la suma por rama y año tiene que dar lo mismo que comercio_por_rama_anio
control = por_uso.groupby(["rama", "anio"]).imp_cif.sum().reset_index().merge(comercio[["rama", "anio", "imp_cif"]], on=["rama", "anio"], suffixes=("", "_total"))
assert ((control.imp_cif / control.imp_cif_total - 1).abs() < 1e-9).all()
por_uso.groupby(["anio", "uso"]).imp_cif.sum().unstack().div(1e9).round(1)

uso,Capital,Consumo,Insumos,Sin clasificar
anio,,,,
2016,12.0,11.3,27.7,0.0
2017,15.3,14.0,31.3,0.0
2018,12.5,12.7,32.0,0.1
2019,8.7,7.8,26.7,0.0
2020,7.6,6.7,22.5,0.0
2021,10.4,7.8,36.4,0.0
2022,12.6,9.4,48.3,0.0
2023,11.2,8.8,41.4,0.0
2024,10.2,9.2,33.5,0.0


In [14]:
resp = A.respuesta_por_rama(panel, E, por_uso)
resp.to_csv(PROCESSED / "respuesta_por_rama.csv", index=False)
resp_grupos = A.respuesta_grupos(resp, I, S)
resp_grupos.to_csv(PROCESSED / "respuesta_grupos.csv", index=False)
resp_corr = A.respuesta_correlaciones(resp)
resp_corr.to_csv(PROCESSED / "respuesta_correlaciones.csv", index=False)
resp_sens = A.respuesta_sensibilidad(resp)
resp_sens.to_csv(PROCESSED / "respuesta_sensibilidad.csv", index=False)
resp_uso_corr = A.respuesta_uso_correlaciones(resp)
resp_uso_corr.to_csv(PROCESSED / "respuesta_uso_correlaciones.csv", index=False)
resp_uso_grupos = A.respuesta_uso_grupos(resp)
resp_uso_grupos.to_csv(PROCESSED / "respuesta_uso_grupos.csv", index=False)
resp_uso_corr.drop(columns=["exposicion_col", "resultado_col"]).round(3)

,exposicion,resultado,rho,p_valor,n_ramas,p_corregido
0,Bienes de consumo (USD),Empleo,0.007,0.980,15,0.980
1,Bienes de consumo (USD),Producción,-0.057,0.840,15,0.980
2,Bienes de consumo (USD),Salario real,-0.300,0.277,15,0.666
3,Bienes de consumo (toneladas),Empleo,0.036,0.899,15,0.980
4,Bienes de consumo (toneladas),Producción,-0.432,0.108,15,0.323
5,Bienes de consumo (toneladas),Salario real,-0.146,0.603,15,0.904
6,Insumos (USD),Empleo,0.779,0.000,16,0.004
7,Insumos (USD),Producción,0.191,0.478,16,0.820
8,Insumos (USD),Salario real,0.618,0.011,16,0.065
9,Bienes de capital (USD),Empleo,0.393,0.383,7,0.767
